# 17.8 — Legge dei grandi numeri

**Prerequisiti:** 17.7 Convergenza di variabili aleatorie, 17.5 Varianza, 17.6 Indipendenza.


## Perché ci interessa

L'intuizione che "ripetendo un esperimento molte volte, la media delle osservazioni si avvicina al valore atteso" è alla base di ogni applicazione pratica della probabilità: dai sondaggi ai casinò, dalla fisica statistica al machine learning. Questo notebook trasforma quell'intuizione in un teorema, usando esattamente gli strumenti costruiti nei notebook precedenti: la disuguaglianza di Chebyshev (notebook 17.5) e l'additività delle varianze per variabili indipendenti (notebook 17.6). È il primo dei due grandi "teoremi limite" della probabilità (l'altro, il teorema centrale del limite, è l'oggetto del notebook 17.9), e mostra come le nozioni di convergenza appena classificate (notebook 17.7) non siano un esercizio astratto ma il linguaggio necessario per enunciare risultati fondamentali.

## Teoria, passo dopo passo

### Impostazione

Siano $X_1,X_2,\ldots$ variabili aleatorie **indipendenti e identicamente distribuite** (i.i.d.): indipendenti nel senso del notebook 17.6, e tutte con la stessa distribuzione, quindi lo stesso valore atteso $\mu=E[X_i]$ e la stessa varianza $\sigma^2=\mathrm{Var}(X_i)$ (supposta finita). Definiamo la **media campionaria**
$$\overline X_n=\frac{X_1+\cdots+X_n}n.$$

### Legge debole dei grandi numeri

**Teorema.** $\overline X_n\to\mu$ in probabilità.

**Dimostrazione.** Per linearità del valore atteso (notebook 17.4): $E[\overline X_n]=\dfrac1nE[X_1+\cdots+X_n]=\dfrac1n\cdot n\mu=\mu$. Per l'additività della varianza per indipendenti (notebook 17.6, applicata ripetutamente a $X_1,\ldots,X_n$) e la proprietà $\mathrm{Var}(aX)=a^2\mathrm{Var}(X)$ (notebook 17.5): $\mathrm{Var}(\overline X_n)=\dfrac1{n^2}\mathrm{Var}(X_1+\cdots+X_n)=\dfrac1{n^2}\cdot n\sigma^2=\dfrac{\sigma^2}n$. Per la disuguaglianza di Chebyshev (notebook 17.5, esercizio 4), per ogni $\varepsilon>0$:
$$P\big(|\overline X_n-\mu|\ge\varepsilon\big)\le\frac{\mathrm{Var}(\overline X_n)}{\varepsilon^2}=\frac{\sigma^2}{n\varepsilon^2}\xrightarrow[n\to\infty]{}0$$
(essendo $\sigma^2,\varepsilon^2$ costanti fissate, il membro destro tende a $0$ come $1/n$, notebook 5.4). Per definizione di convergenza in probabilità (notebook 17.7), $\overline X_n\to\mu$ in probabilità. $\blacksquare$

**Perché "debole".** Il nome distingue questo risultato dalla legge **forte**, che afferma la convergenza quasi certa (una nozione strettamente più forte, notebook 17.7) — un risultato più profondo, che richiede tecniche oltre lo scopo dimostrativo di questo corso.

### Legge forte dei grandi numeri (enunciato)

**Teorema (Kolmogorov, senza dimostrazione completa).** Se $X_1,X_2,\ldots$ sono i.i.d. con $E[|X_i|]<\infty$ (bastano momenti primi finiti, non serve la varianza finita), allora $\overline X_n\to\mu$ **quasi certamente**.

La dimostrazione rigorosa di questo teorema richiede strumenti (il lemma di Borel-Cantelli in una forma più raffinata di quella vista nel notebook 17.1, e tecniche di troncamento) che esulano dallo scopo introduttivo di questo modulo: lo enunciamo qui perché è il risultato che giustifica rigorosamente l'interpretazione **frequentista** della probabilità stessa — "$P(A)$ è il limite, quasi certo, della frequenza relativa di $A$ in prove ripetute indipendenti" — chiudendo concettualmente il cerchio aperto nel notebook 17.1: gli assiomi di Kolmogorov, che a prima vista sembravano solo una scelta formale elegante, producono esattamente il comportamento empirico che la parola "probabilità" evoca intuitivamente.

**Un'osservazione utile.** Sotto l'ipotesi più forte $E[X_i^2]<\infty$ (varianza finita), si può in realtà dimostrare la legge forte con strumenti alla nostra portata (una versione del lemma di Borel-Cantelli applicata a $\overline X_{n^2}$ lungo la sottosuccessione dei quadrati, più un argomento di controllo tra un quadrato e il successivo), ma anche questa dimostrazione più elementare richiede più spazio di quanto sia opportuno dedicarle qui: ci limitiamo a enunciare il risultato generale e a verificarlo numericamente.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Verifica numerica della legge (debole e, empiricamente, anche forte) dei
# grandi numeri: media campionaria di variabili Exp(1) (mu=1) al crescere di n.

np.random.seed(5)
n_max = 20000
campioni = np.random.exponential(scale=1.0, size=n_max)
medie_cumulative = np.cumsum(campioni) / np.arange(1, n_max+1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(medie_cumulative, linewidth=0.8)
ax.axhline(1.0, color='red', linestyle='--', label='mu = 1')
ax.set_xlabel("n")
ax.set_ylabel("media campionaria X_n barrata")
ax.set_title("Legge dei grandi numeri: la media campionaria converge a mu")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Media campionaria dopo n=100: {medie_cumulative[99]:.4f}")
print(f"Media campionaria dopo n=1000: {medie_cumulative[999]:.4f}")
print(f"Media campionaria dopo n=20000: {medie_cumulative[-1]:.4f}")
print("(valore teorico mu = 1)")

# Verifica quantitativa della legge debole: P(|X_n_barra - mu| >= eps) decresce come 1/n
eps = 0.05
n_test = [100, 500, 2000, 10000]
n_ripetizioni = 5000
for n in n_test:
    campioni_ripetuti = np.random.exponential(scale=1.0, size=(n_ripetizioni, n))
    medie = campioni_ripetuti.mean(axis=1)
    P_stimata = np.mean(np.abs(medie - 1.0) >= eps)
    bound_chebyshev = 1.0 / (n * eps**2)  # sigma^2=1 per Exp(1)
    print(f"n={n}: P(|X_barra-mu|>={eps}) stimata = {P_stimata:.4f}, bound Chebyshev = {min(bound_chebyshev,1):.4f}")

## Esercizio svolto e commentato

**Problema:** usando la disuguaglianza di Chebyshev, trova quante prove Bernoulli($p=0.5$) indipendenti servono per garantire $P(|\overline X_n-0.5|\ge0.01)\le0.05$.

**Ragionamento:**

**Passo 1 — richiamiamo $\sigma^2$ per una Bernoulli.** $E[X_i]=p=0.5$. $\mathrm{Var}(X_i)=E[X_i^2]-E[X_i]^2=p-p^2=p(1-p)=0.5\cdot0.5=0.25$ (usando $X_i^2=X_i$ per una variabile a valori in $\{0,1\}$).

**Passo 2 — applichiamo il bound di Chebyshev dalla dimostrazione della teoria.** $P(|\overline X_n-\mu|\ge\varepsilon)\le\dfrac{\sigma^2}{n\varepsilon^2}=\dfrac{0.25}{n\cdot(0.01)^2}=\dfrac{0.25}{0.0001\,n}=\dfrac{2500}n$.

**Passo 3 — imponiamo che il bound sia $\le0.05$.** $\dfrac{2500}n\le0.05\iff n\ge\dfrac{2500}{0.05}=50000$.

**Conclusione:** servono almeno $n=50000$ prove per garantire (secondo la stima, pessimistica, di Chebyshev) che la media campionaria si discosti da $0.5$ per più di $0.01$ con probabilità al più $5\%$. Questo numero è **grande**: la disuguaglianza di Chebyshev, essendo valida per **ogni** distribuzione con varianza finita (nessuna ipotesi aggiuntiva), è necessariamente una stima molto conservativa — il notebook 17.9 (teorema centrale del limite) fornirà un'approssimazione molto più precisa, sfruttando la forma specifica della distribuzione limite invece di limitarsi a un bound universale.

In [ ]:
import numpy as np

p = 0.5
sigma2 = p*(1-p)
eps = 0.01
delta = 0.05

n_necessario = sigma2 / (eps**2 * delta)
print(f"n necessario secondo Chebyshev: {n_necessario:.0f}")

# Verifica empirica: con n=50000, quale e' la probabilita' EFFETTIVA di scostamento?
np.random.seed(6)
n = 50000
n_ripetizioni = 20000
prove = np.random.binomial(1, p, size=(n_ripetizioni, n))
medie = prove.mean(axis=1)
P_effettiva = np.mean(np.abs(medie - p) >= eps)
print(f"P(|X_barra - 0.5| >= 0.01) EFFETTIVA (simulata): {P_effettiva:.5f}")
print(f"Bound di Chebyshev: {delta}")
print("La probabilita' reale e' molto piu' piccola del bound: Chebyshev e' molto conservativo.")

## Esercizi da risolvere

1. Ripeti il calcolo dell'esercizio svolto per $\varepsilon=0.05$ (invece di $0.01$), mantenendo $\delta=0.05$: di quanto diminuisce $n$?
2. Dimostra che, se $X_1,\ldots,X_n$ sono i.i.d. con $\mathrm{Var}(X_i)=\sigma^2$, allora $\mathrm{Var}\big(\overline X_n\big)\to0$ per $n\to\infty$, e interpreta questo fatto in relazione alla legge dei grandi numeri (suggerimento: rileggi la dimostrazione della teoria).
3. Usa la legge dei grandi numeri per giustificare informalmente il metodo Monte Carlo: per stimare $E[g(X)]$ per una variabile $X$ di cui sappiamo campionare, si genera un campione i.i.d. $X_1,\ldots,X_n$ e si calcola $\frac1n\sum_ig(X_i)$. Spiega perché questa media converge (in probabilità) a $E[g(X)]$, applicando la legge dei grandi numeri alla successione $Y_i=g(X_i)$.
4. *(più impegnativo)* La disuguaglianza di Chebyshev usa solo il secondo momento. Dimostra una versione più forte usando il quarto momento: se $E[X_i^4]<\infty$, allora (con un argomento analogo a quello di Chebyshev, applicato a $Z=(\overline X_n-\mu)^4$ e $t=\varepsilon^4$ nella disuguaglianza di Markov) $P(|\overline X_n-\mu|\ge\varepsilon)\le\dfrac{E[(\overline X_n-\mu)^4]}{\varepsilon^4}$, e spiega (senza calcolare esplicitamente $E[(\overline X_n-\mu)^4]$, un conto che coinvolge l'espansione del binomio alla quarta potenza) perché ci si aspetta che questo bound decada come $1/n^2$ invece di $1/n$, dando una stima più precisa per $n$ grande.

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $n\ge\dfrac{\sigma^2}{\varepsilon^2\delta}=\dfrac{0.25}{(0.05)^2\cdot0.05}=\dfrac{0.25}{0.000125}=2000$. Molto meno di $50000$: la richiesta di precisione $\varepsilon$ entra nel bound come $1/\varepsilon^2$, quindi rilassare la precisione di un fattore $5$ (da $0.01$ a $0.05$) riduce $n$ di un fattore $25$.

**2.** Dalla dimostrazione della teoria, $\mathrm{Var}(\overline X_n)=\sigma^2/n$. Essendo $\sigma^2$ una costante fissata e $n\to\infty$, $\sigma^2/n\to0$ (notebook 5.4). Interpretazione: la media campionaria non solo ha lo stesso valore atteso $\mu$ per ogni $n$ (fatto già vero per $n=1$), ma la sua **dispersione attorno a $\mu$ si restringe** sempre di più al crescere di $n$ — è proprio questo restringimento della varianza, tramite Chebyshev, a forzare la concentrazione di probabilità (quindi la convergenza) attorno a $\mu$.

**3.** Le $Y_i=g(X_i)$ sono i.i.d. (essendo $X_i$ i.i.d. e $g$ la stessa funzione applicata a ciascuna, un fatto analogo all'esercizio 2 del notebook 17.6 esteso a più variabili), con $E[Y_i]=E[g(X)]$ (stessa distribuzione di $X$) e varianza finita se $E[g(X)^2]<\infty$. Per la legge dei grandi numeri applicata a $Y_1,\ldots,Y_n$: $\overline Y_n=\frac1n\sum_ig(X_i)\to E[g(X)]$ in probabilità. Questo giustifica il metodo Monte Carlo: la media campionaria di valutazioni ripetute di $g$ su campioni indipendenti di $X$ converge (in probabilità, con abbastanza campioni quasi certamente per la legge forte) al valore atteso cercato — esattamente la tecnica usata in tutte le celle di verifica computazionale di questo modulo.

**4.** Per la disuguaglianza di Markov applicata a $Z=(\overline X_n-\mu)^4\ge0$ con $t=\varepsilon^4$ (stessa dimostrazione del notebook 17.5, esercizio 4, con l'esponente $4$ al posto di $2$): $P\big((\overline X_n-\mu)^4\ge\varepsilon^4\big)\le\dfrac{E[(\overline X_n-\mu)^4]}{\varepsilon^4}$, ed essendo $(\overline X_n-\mu)^4\ge\varepsilon^4\iff|\overline X_n-\mu|\ge\varepsilon$ (radice quarta crescente su $[0,\infty)$): la disuguaglianza segue. Per l'ordine di grandezza: $\overline X_n-\mu=\frac1n\sum_i(X_i-\mu)$, quindi $(\overline X_n-\mu)^4$ coinvolge, espandendo, termini come $(X_i-\mu)^4/n^4$ (ce ne sono $n$, contribuendo $O(1/n^3)$ in totale) e termini incrociati come $(X_i-\mu)^2(X_j-\mu)^2/n^4$ per $i\ne j$ (che, essendo le $X_i$ indipendenti a media nulla dopo la traslazione, non si annullano al livello del quarto momento come accadeva per la varianza, ma restano comunque $O(n^2)$ termini, ciascuno $O(1/n^4)$, contribuendo $O(1/n^2)$ in totale): il termine dominante è $O(1/n^2)$, non $O(1/n)$ come per Chebyshev — un bound che decade più velocemente, quindi più informativo per $n$ grande, al prezzo di richiedere l'esistenza del quarto momento invece del solo secondo.

</details>

## Riepilogo finale

- La **legge debole dei grandi numeri** dimostra che $\overline X_n\to\mu$ in probabilità per variabili i.i.d. con varianza finita, usando solo la disuguaglianza di Chebyshev (notebook 17.5) e l'additività delle varianze per indipendenti (notebook 17.6): la dimostrazione più semplice possibile per un risultato fondamentale.
- La **legge forte**, enunciata senza dimostrazione completa, afferma la convergenza quasi certa sotto la sola ipotesi di primo momento finito: è questo risultato più profondo a giustificare rigorosamente l'interpretazione frequentista della probabilità.
- Il bound di Chebyshev, valido per ogni distribuzione con varianza finita, è tipicamente molto conservativo rispetto al comportamento reale, come mostra la verifica numerica.
- Il metodo Monte Carlo — la base computazionale usata implicitamente in ogni cella di verifica di questo modulo — è un'applicazione diretta della legge dei grandi numeri.
- Bound più raffinati (momenti di ordine superiore) danno stime più precise, al prezzo di ipotesi più forti: un compromesso tipico in probabilità e statistica.

## Collegamenti

Il prossimo notebook, **17.9 Teorema centrale del limite**, va oltre la semplice convergenza: descrive con precisione **come** $\overline X_n$ si avvicina a $\mu$, mostrando che le fluttuazioni, opportunamente riscalate, seguono sempre una distribuzione normale — indipendentemente dalla distribuzione di partenza delle $X_i$.